In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from src.downloadData import download_dataset_csvs
summary = download_dataset_csvs(
    output_dir=repo_root /"data" /"raw",
    progress=True,
)

print(f"downloaded={len(summary.downloaded_paths)}, skipped={len(summary.skipped_paths)}")

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType

import os
import sys
import shutil

python_exe = sys.executable
os.environ["PYSPARK_PYTHON"] = python_exe
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exe

spark = (
    SparkSession.builder
    .appName("EEG_Schizoprenia")
    .master("local[*]")
    .config("spark.pyspark.python", python_exe)
    .config("spark.pyspark.driver.python", python_exe)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark started:", spark.version)
print("Python:", sc.pythonExec)

In [ ]:
# Cell 2: Config
INPUT_ROOT = Path("small/raw")
OUTPUT_PATH = "small/processed/subject_csv_means"
EXPECTED_IDS = range(1, 82)   # 1..81
COLUMN_COUNT = 74             # based on your current files
COLUMN_NAMES = [f"col_{i}" for i in range(1, COLUMN_COUNT + 1)]

def subject_path(i: int) -> Path:
    return INPUT_ROOT / f"{i}.csv" / f"{i}.csv"

paths = [subject_path(i) for i in EXPECTED_IDS if subject_path(i).exists()]
missing = [subject_path(i) for i in EXPECTED_IDS if not subject_path(i).exists()]

print(f"Found {len(paths)} files")
print(f"Missing {len(missing)} files")


In [ ]:
# Cell 3: Per-file averaging
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType


schema = StructType([StructField(c, StringType(), True) for c in COLUMN_NAMES])

rows = []

for idx, path in enumerate(paths, start=1):
    print(f"[{idx}/{len(paths)}] {path}")
    
    df = (
        spark.read
        .option("header", "false")
        .option("mode", "PERMISSIVE")
        .schema(schema)
        .csv(str(path))
    )
    
    numeric_df = df.select(*[
        F.when(F.length(F.trim(F.col(c))) == 0, None)
         .otherwise(F.trim(F.col(c)).cast("double"))
         .alias(c)
        for c in COLUMN_NAMES
    ])
    
    agg = numeric_df.agg(*[
        F.avg(F.col(c)).alias(f"mean_{c}")
        for c in COLUMN_NAMES
    ]).collect()[0].asDict()
    
    agg["file_id"] = path.stem
    rows.append(agg)


In [ ]:
# Cell 4: Final output
result_df = spark.createDataFrame(rows).select(
    "file_id", *[f"mean_{c}" for c in COLUMN_NAMES]
)

result_df.orderBy("file_id").show(5, truncate=False)
print("Output rows:", result_df.count())

result_df.write.mode("overwrite").parquet(OUTPUT_PATH)


In [ ]:
# Cell 5: Optional CSV output instead of Parquet
result_df.coalesce(1).write.mode("overwrite").option("header", True).csv("data/.processed/test")

In [ ]:
spark.stop()